# 密闭演示盒：光源标定

配合 `docs/04_demo_box.md` 用。这个 notebook 只做一件事：
**把手动曝光打开，然后帮你把盒子的三档灯位标定到原来的三个类别中心。**

## 为什么必须切手动曝光

默认寄存器表里 `0x3503 = 0x00`，也就是 AEC（自动曝光）+ AGC（自动增益）全开。
在盒子里调暗灯时，摄像机会自动把曝光拉回去补偿，画面亮度被顶回正常 ——
于是 MLP 看到的还是「常规」场景，判不出暗光。

实测证据：遮镜头时 `f_luma` 最低只降到 **27**，而暗光类的中心是 **10**。
不是遮得不够，是 AEC 在补偿。

切手动之后，画面亮度**完全由你的灯决定**，三档可以精确标定、重复性接近 100%。

## 跑之前

- [ ] 盒子内壁贴黑、镜头孔密封（漏光会让暗光档卡在 27）
- [ ] 灯是**直流调光**，不是低频 PWM（否则画面出滚动条纹，污染 f_grad）
- [ ] 三档目标物准备好了：剪影卡（逆光）、网格/文字卡（常规）


In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

# --- 自动定位工程根目录（和其它 notebook 一样）-----------------------------
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib import exposure
from lib.ai_video_regs import AiVideoRegs

# ===========================================================================
#  盒子模式的曝光工作点：曝光固定，亮度完全交给你的灯。
#
#  这两个值**必须和采集/演示用同一档** —— 单一来源是 lib/exposure.py 的 WORKPOINT。
#  不这么做的话：你在标定里把灯"调到位"，换到演示 notebook 上亮度又变了，
#  特征跟着跑掉、MLP 判错，而且很难想到是曝光这一层的问题。
#
#  下面「扫描曝光 / 增益」那两格只是用来**确认**这一档对你的盒子合适；
#  真要改工作点，改 lib/exposure.py 的 WORKPOINT（一处生效），别只改这里。
# ===========================================================================
EXPOSURE_LINES = exposure.WORKPOINT["exposure"]   # 736 行（采集/演示同款）
GAIN_CODE      = exposure.WORKPOINT["gain"]       # 0x20（采集/演示同款）

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)
vs.stop_all_channels(vdma0, vdma1)

# ---- 切手动曝光（这一步不做，后面调灯全白费）------------------------------
exposure.set_manual(iic, exposure=EXPOSURE_LINES, gain=GAIN_CODE)
exposure.report(iic)

regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X  (硬件阶段 %d)" % (regs.version(), regs.version_stage()))
assert regs.version_stage() >= 7, "本 notebook 需要阶段 7 的硬件"

vp.start_capture(vdma0, mode)
regs.wait_new_frame()
print()
print("采集已启动。先把灯开到『常规档』的水平，再跑下一格看 f_luma。")


## 三档目标值

下面这三行**不是写死的**，跟 `data/features.csv` 现算（540 帧 = 每类 180 帧实测均值，2026-10-13 重采）：

| 档位 | 目标 f = (luma, luma2, grad) | 类内散布 std | 关键差别 |
|------|------------------------------|--------------|---------|
| 暗光 | (9.7, 0.2, 17.2)  | (4.0, 0.4, 4.1) | 亮度最低 |
| 逆光 | (66.4, 44.6, 21.4) | (9.5, 7.0, 1.7) | **对比度最高**（靠明暗并存，不是靠亮；`luma2` 必须上去） |
| 常规 | (96.5, 68.6, 29.9) | (9.5, 6.2, 1.0) | **纹理最多**（要有细节，白墙不行） |

三类中心两两距离：0↔1 = 72.2、0↔2 = 111.2、**1↔2 = 39.4**（最近的一对是逆光和常规）。

标定的目标就是让盒子的三档**命中这三个点** —— 命中了，现有的 MLP 模型就直接可用，不用重训。
这三行只是给人看的对照表；下面那格是**从 features.csv 读**，数据一改它就跟着变，所以你不用回来手改数字。


In [ ]:
# 三类中心 / 散布 —— 从 data/features.csv 现场算（和报告、lib/adaptive.py 同一来源）
# 千万别写死：数据重采之后中心会动（旧中心 9.8/67.4/109.3 → 现中心 9.7/66.4/96.5），
# 写死的后果很隐蔽 —— 你按旧目标把灯调"到位"，特征其实落在训练分布外面，MLP 照样判错。
from lib import adaptive as ad

CLASS_MEAN = {k: tuple(ad.CLASS_MEAN[k]) for k in (0, 1, 2)}
CLASS_STD = {k: tuple(ad.CLASS_STD[k]) for k in (0, 1, 2)}
CLASS_NAME = {k: ad.CLASS_NAME[k] for k in (0, 1, 2)}

print("三类中心来源：%s" % ad.CLASS_STATS_SOURCE)
for k in (0, 1, 2):
    print("  %d %-6s 目标 f=(%5.1f,%5.1f,%5.1f)  类内 std=(%.1f,%.1f,%.1f)"
          % (k, CLASS_NAME[k], CLASS_MEAN[k][0], CLASS_MEAN[k][1], CLASS_MEAN[k][2],
             CLASS_STD[k][0], CLASS_STD[k][1], CLASS_STD[k][2]))


def health(tag=""):
    """读一次：MLP 判决 + 三个特征 + 到三类中心的距离。调灯时盯着它。"""
    st = regs.raw_stats()
    f1, f2, f3 = regs.mlp_features_from_raw(st)
    luma, contrast, sharp = regs.features_from_raw(st)
    dist = {k: ((f1 - a) ** 2 + (f2 - b) ** 2 + (f3 - c) ** 2) ** 0.5
            for k, (a, b, c) in CLASS_MEAN.items()}
    near = min(dist, key=dist.get)
    print("---- %s ----" % tag)
    print("MLP=%d  最近类别=%d(%s)  MODE_SEL=%d  FRAME_CNT=%d"
          % (regs.mlp_output(), near, CLASS_NAME[near],
             regs.mode_sel(), regs.frame_cnt()))
    print("f=(%3d,%3d,%3d)   luma=%6.1f contrast=%6.1f sharpness=%6.1f"
          % (f1, f2, f3, luma, contrast, sharp))
    print("距离: " + "   ".join("%s=%.1f" % (CLASS_NAME[k], v)
                               for k, v in sorted(dist.items())))
    return dict(f=(f1, f2, f3), near=near, mlp=regs.mlp_output())


health("读一次看看")


## 光源闪烁 / 条纹检测（做盒子之前先跑这一格）

**手机拍灯会闪，不代表我们这个相机一定会出条纹** —— 两者的曝光时间差很多：手机的曝光通常只有几毫秒，而这里的曝光是 512 行 ≈ 22 ms，足以把 100 Hz 的纹波平均掉（22 ms 覆盖 2 个多周期）。

所以**先实测，别急着换灯**。

判据：

* 行平均亮度曲线是**平滑**的（只有缓慢起伏）→ 没问题，直接去标定；
* 出现**规则的周期性起伏**（正弦/方波状）→ 灯的 PWM 频率太低，看下面的补救顺序。

三种补救，都不花钱：

1. **把灯整体调暗、同时把曝光加大**（亮度保持不变，但纹波被平均掉）。注意是「灯暗 + 曝光长」，不是「灯亮 + 曝光短」。
2. 灯前加几层硫酸纸 / 中性灰片继续减光，就能换更长的曝光。
3. 还不行 → 换灯（LED 灯带 + 直流调光模块，约 30 元）。

> 进阶（能拆开台灯才行）：在 LED 两端并联一个 1000 µF / 16 V 电解电容，把 PWM 电流纹波平滑掉。密封的台灯通常够不到驱动输出，就用上面三步。


In [ ]:
# ---- 条纹检测：抓一帧，看「每一行的平均亮度」曲线 ----
raw = vp.capture_frame(vdma0)
rgb = vp.frame_to_rgb(raw)
gray = rgb.mean(axis=2) if rgb.ndim == 3 else rgb.astype(float)
row_mean = gray.mean(axis=1)

bright = row_mean.mean()
print("整幅平均亮度 = %.1f   每行均值标准差 = %.2f" % (bright, row_mean.std()))
if bright < 80:
    print("! 太暗了：平均亮度只有 %.1f。暗场测不出条纹，f_grad 也会被量化卡死。" % bright)
    print("  请把灯开到『常规档』（整幅平均亮度 100~130）再跑这一格。")

plt.figure(figsize=(12, 3))
plt.plot(row_mean, linewidth=0.8)
plt.xlabel("row index")
plt.ylabel("row mean brightness")
plt.title("row-mean brightness: periodic ripple = PWM banding")
plt.grid(True)
plt.show()

# 再连读 10 次 f_grad：条纹会让清晰度这个特征跳得很厉害
fg = [regs.mlp_features()[2] for _ in range(10)]
print("f_grad x10:", fg, "  spread =", max(fg) - min(fg))
print()
print("判据（亮度足够时才有意义）：")
print("  * 曲线平滑 + f_grad 极差 <= 2      -> 光源没问题，继续标定")
print("  * 曲线有等间距的周期起伏 -> PWM 条纹，按上面三步补救")


## 调灯助手（实时）

跑下面这格，然后**一边拧灯一边看输出**。目标是让三档分别落在目标附近。

提示：先不管 MLP 判成什么，只看 `f=(...)` 和 `最近=...` 两列。


In [ ]:
def observe(seconds=20.0, poll=0.4):
    t0 = time.time()
    while time.time() - t0 < seconds:
        st = regs.raw_stats()
        f = regs.mlp_features_from_raw(st)
        dist = {k: ((f[0] - a) ** 2 + (f[1] - b) ** 2 + (f[2] - c) ** 2) ** 0.5
                for k, (a, b, c) in CLASS_MEAN.items()}
        near = min(dist, key=dist.get)
        print("f=(%3d,%3d,%3d)  MLP=%d  最近=%-5s(%.0f)  MODE=%d"
              % (f[0], f[1], f[2], regs.mlp_output(),
                 CLASS_NAME[near], dist[near], regs.mode_sel()))
        time.sleep(poll)


if "CLASS_MEAN" not in globals():
    raise RuntimeError("请先跑第 4 格（定义 CLASS_MEAN / health），再跑这一格")

# 跑起来之后，用手/旋钮调灯
observe(20.0)


## 扫描曝光 / 增益（选固定工作点）

**先把灯固定在一个不变的位置**，再跑下面两格。

* 曝光太小 → 三个档都偏暗，怎么调灯都上不去；
* 曝光太大 → 常规档过曝（`f_luma` 顶到上限，调灯没分辨率）。

原则：**常规档的目标物 + 主光源中亮** 时，`f_luma` 落在 **96（目标中心）** 最好。

工作点已经固定成采集/演示那一档（736 行 + 增益 0x20），下面两格是**复核**用的：
如果 736 行的读数离目标很远，要调的是**灯**，不是这个工作点。


In [ ]:
def f_luma_now(n=3):
    """等 n 帧再读 f_luma，避开正在统计中的那一帧。"""
    for _ in range(n):
        regs.wait_new_frame()
    return regs.mlp_features()[0]


print("== 曝光扫描（灯固定不动！）==")
print(" 曝光(行)   f_luma")
for e in (64, 128, 192, 256, 384, 512, 640, 736):
    exposure.set_exposure(iic, e)
    time.sleep(0.3)
    print("  %5d     %5d" % (e, f_luma_now()))
exposure.set_exposure(iic, EXPOSURE_LINES)   # 扫完回到工作点，别让后面的 record() 读歪

print()
print("正常结果：736 行应该就是采集时那一档（常规档 f_luma ≈ 96）。")
print("如果 736 行离 96 很远，先把灯调到常规档再扫一遍 —— 别改工作点。")
print("真要改工作点：改 lib/exposure.py 的 WORKPOINT，然后重采数据（否则训练分布对不上）。")


In [ ]:
print("== 增益扫描（灯固定不动！）==")
print(" 增益码    f_luma")
for g in (0x10, 0x18, 0x20, 0x30, 0x40):
    exposure.set_gain(iic, g)
    time.sleep(0.3)
    print("  0x%02X     %5d" % (g, f_luma_now()))
exposure.set_gain(iic, GAIN_CODE)            # 扫完回到工作点（0x20）

print()
print("采集/演示用的是 0x20；这里只做复核（同样的灯，看增益对 f_luma / f_grad 的影响）。")
print("注意：增益同时放大噪声，会把 f_grad 抬上去、让『常规』和『逆光』的区分变差。")
print("做完这一格记得把增益设回 0x%02X（重跑第 1 格最省事）。" % exposure.WORKPOINT["gain"])


## 三档记录 + 达标判定

按顺序做三遍：**调灯 → 调用一次 `record(...)`**。

```python
record("暗光")   # 主光源关到最暗
record("逆光")   # 后壁光板开、前面放剪影
record("常规")   # 主光源开到中亮、放网格卡
```

三档都记完，再跑 `check()`。


In [ ]:
SAMPLE = {}


def record(label):
    """把当前这一档的读数记下来。调好灯之后调用一次。"""
    st = regs.raw_stats()
    f = regs.mlp_features_from_raw(st)
    SAMPLE[label] = f
    d = {k: ((f[0] - a) ** 2 + (f[1] - b) ** 2 + (f[2] - c) ** 2) ** 0.5
         for k, (a, b, c) in CLASS_MEAN.items()}
    near = min(d, key=d.get)
    print("记录 %-4s  f=(%3d,%3d,%3d)  MLP=%d  最近=%s"
          % (label, f[0], f[1], f[2], regs.mlp_output(), CLASS_NAME[near]))
    return f


# ===== 按顺序做：调灯 -> record =====
# record("暗光")
# record("逆光")
# record("常规")
record("先随便记一次，看看格式")


In [ ]:
TARGET = {"暗光": 0, "逆光": 1, "常规": 2}


def check():
    if len(SAMPLE) < 3:
        print("还没记满三档。先跑上面那格，把 record('暗光'/'逆光'/'常规') 各调一次。")
        return False
    print("%-4s %-16s %-16s %-8s %s" % ("档位", "实测 f", "目标 f", "距离", "判定"))
    ok = True
    for label, cls in TARGET.items():
        f = SAMPLE[label]
        t = CLASS_MEAN[cls]
        d = ((f[0] - t[0]) ** 2 + (f[1] - t[1]) ** 2 + (f[2] - t[2]) ** 2) ** 0.5
        good = d < 20.0        # 三类中心最近的一对相距 45，20 是宽松但够用的判定线
        ok = ok and good
        print("%-4s %-16s %-16s %8.1f %s"
              % (label, "(%d,%d,%d)" % f, "(%d,%d,%d)" % t, d,
                 "OK" if good else "偏差大"))
    print()
    if ok:
        print("三档全部命中 ✅  现有模型和 bitstream 直接可用，不用重训。")
    else:
        print("还有档位没命中 ❌  按上面『距离』最大的那一档重新调灯，再 record 一次。")
    return ok


check()


## 三档画面差异（肉眼确认）

标定完顺手看一眼三档的处理效果 —— mode 0 的 ×1.5 最明显，
mode 1 是"压对比 / 发灰"，mode 2 是"变锐"。演示时就用 0 ↔ 2 做对比最抓眼。


In [ ]:
MODE_NAMES = ["0 dark-boost x1.5", "1 backlight-compress", "2 detail-sharpen"]
shots = {}

for m in (0, 1, 2):
    regs.set_mode(m)
    for _ in range(3):          # 等 3 帧，确保 mode_active 已在帧边界更新
        regs.wait_new_frame()
    shots[m] = vp.frame_to_rgb(vp.capture_frame(vdma0))
    print("mode %d (%-20s) 均值亮度 = %.1f" % (m, MODE_NAMES[m], shots[m].mean()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, (0, 1, 2)):
    ax.imshow(shots[m])
    ax.set_title("MODE_SEL=%d  %s" % (m, MODE_NAMES[m]))
    ax.axis("off")
plt.tight_layout()
plt.show()

print("平均亮度: " + ", ".join("mode%d=%.1f" % (m, shots[m].mean()) for m in (0, 1, 2)))
print("（mode0 应该明显比 mode2 亮；不成立说明像素通路没接对）")


## 收尾

恢复自动曝光、释放资源。**不恢复的话**，下一个 notebook 会继续用手动曝光，
普通演示的判定可能就不对了。


In [ ]:
# 模式先恢复到 0，方便下次演示
try:
    regs.set_mode(0)
except Exception as e:
    print("[warn] 恢复模式失败：%s" % e)

# 恢复自动曝光（普通演示模式）
try:
    exposure.set_auto(iic)
except Exception as e:
    print("[warn] 恢复自动曝光失败：%s" % e)

# 释放资源
try:
    vs.stop_all_channels(vdma0, vdma1)
    print("[ok] VDMA 通道已停")
except Exception as e:
    print("[warn] 停 VDMA 失败：%s" % e)

try:
    ol.free()
    print("[ok] overlay 已释放")
except Exception as e:
    print("[warn] 释放 overlay 失败：%s" % e)

print()
print("下一步：按 docs/04_demo_box.md 第 6 節，在演示 notebook 里带")
print("manual_exposure=dict(exposure=%d, gain=0x%02X) 重跑。" % (EXPOSURE_LINES, GAIN_CODE))
